# Koya Gait – fast cycle-level pipeline (no sliding window)

**Why the old notebook was slow and inaccurate**

Your CSV has 1,464,925 rows, but each row is only *one time-step* of a gait cycle (35 rows = 1 cycle).
The old notebook treated every row as an independent sample, so models had to guess one of 390 people from a **single
instant** of the signal (that is why Naive Bayes got ~0.21), while training on 1.46 M rows × 390 classes
(+ oversampling + 5-fold + SVC on 1.4 M rows, which is O(n²)).

**What this notebook does instead**

* Reshapes the CSV into **41,855 cycles × 35 time-steps × 12 features**. The cycles are already segmented by `koya.py`, so **no sliding window is needed**.
* Classical ML on the flattened cycle (420 values + 48 summary stats), which is 35× fewer samples.
* Deep learning (CNN / LSTM / CNN-LSTM / GRU) in **PyTorch on your RTX 2050** with mixed precision, with all data kept on the GPU.
* Splits **by recording session** (train sessions 1–7, val 8, test 9–10), which is the honest protocol: neighbouring cycles from the same recording overlap and a random split leaks them into both train and test.

In [ ]:
import os, io, time, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score
warnings.filterwarnings("ignore")

# ---------- paths ----------
DATA_DIR = Path(r"D:\Gait Dataset\Code\Koya_Gait_Combined_12Feature_Output")
CSV   = DATA_DIR / "Koya_Gait_Combined_12Features.csv"
IDX   = DATA_DIR / "Koya_Gait_Cycle_Index.csv"
CACHE = DATA_DIR / "koya_cycles.npz"          # created on the first run -> later loads take ~1 s

# ---------- settings ----------
FEATURES = ["accx","accy","accz","acc_mag","gyx","gyy","gyz","gy_mag","magx","magy","magz","mag_mag"]
T        = 35            # timesteps per cycle (dataset_summary.json)
SEED     = 42
SPLIT_MODE = "session"   # "session" (recommended) or "random" (stratified, leaks overlapping cycles)

N_TREES  = 100           # Random Forest / Extra Trees
RUN_XGB  = False         # 390 classes -> 390 trees per boosting round; enable only if you can wait
RUN_KFOLD = False        # optional session-grouped 5-fold (see end of notebook)

## 1. Load the CSV once and reshape to cycles

In [ ]:
if CACHE.exists():
    z = np.load(CACHE)
    X, y, session = z["X"], z["y"], z["session"]
else:
    dtypes = {c: np.float32 for c in FEATURES}; dtypes["Label"] = np.int32
    df  = pd.read_csv(CSV, dtype=dtypes)                       # float32 = half the RAM, faster parse
    idx = pd.read_csv(IDX, usecols=["Label", "Session"])

    assert len(df) % T == 0, "rows are not a multiple of 35"
    X   = df[FEATURES].to_numpy(np.float32).reshape(-1, T, len(FEATURES))   # (cycles, 35, 12)
    lab = df["Label"].to_numpy().reshape(-1, T)
    assert (lab == lab[:, :1]).all(), "a cycle contains more than one label"
    y = lab[:, 0] - 1                                                       # 0..389
    assert len(idx) == len(X) and (idx["Label"].to_numpy() - 1 == y).all()
    session = idx["Session"].to_numpy()
    np.savez(CACHE, X=X, y=y, session=session)
    del df

N_CLASSES = int(y.max() + 1)
print("X (cycles, timesteps, features):", X.shape)
print("classes:", N_CLASSES, "| cycles per class min/median/max:",
      np.bincount(y).min(), int(np.median(np.bincount(y))), np.bincount(y).max())
print("NaNs:", int(np.isnan(X).sum()))

## 2. Split + scaling
Class sizes are only 60–154 cycles per person, so the data is nearly balanced and **oversampling is not needed**
(it just duplicated rows and made everything slower). Scaling statistics are fitted on the training part only.

In [ ]:
from sklearn.model_selection import train_test_split
all_idx = np.arange(len(y))

if SPLIT_MODE == "session":
    idx_train = all_idx[session <= 7]
    idx_val   = all_idx[session == 8]
    idx_test  = all_idx[session >= 9]
else:
    idx_trval, idx_test = train_test_split(all_idx, test_size=0.2, random_state=SEED, stratify=y)
    idx_train, idx_val  = train_test_split(idx_trval, test_size=0.125, random_state=SEED, stratify=y[idx_trval])

print(f"train {len(idx_train)} | val {len(idx_val)} | test {len(idx_test)}")
missing = set(range(N_CLASSES)) - set(np.unique(y[idx_train]))
print("classes missing from train:", len(missing))

# per-channel standardisation (train statistics only)
mu = X[idx_train].reshape(-1, X.shape[2]).mean(0)
sd = X[idx_train].reshape(-1, X.shape[2]).std(0) + 1e-8
Xs = ((X - mu) / sd).astype(np.float32)                    # (N, 35, 12) -> used by deep learning

# tabular view for classical ML: 420 raw values + mean/std/min/max per channel
stats = np.concatenate([Xs.mean(1), Xs.std(1), Xs.min(1), Xs.max(1)], axis=1)
X_tab = np.concatenate([Xs.reshape(len(Xs), -1), stats], axis=1).astype(np.float32)
print("tabular:", X_tab.shape)

results = []     # one dict per model -> final comparison table

## 3. Classical ML on cycles (fast)
* Models are trained on **train + val** sessions and scored on the test sessions.
* Random Forest / Extra Trees with 390 classes store ~200 MB **per tree** when fully grown, so 100 trees would need ~20 GB of RAM
  (probably the main reason your old run crawled). `chunked_forest` trains a few trees at a time, accumulates their votes and frees them, so RAM stays small.

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.decomposition import PCA
from sklearn.svm import SVC

idx_fit = np.concatenate([idx_train, idx_val])
Xf, yf = X_tab[idx_fit], y[idx_fit]
Xt, yt = X_tab[idx_test], y[idx_test]

def log(name, pred, secs, family="Classical"):
    acc = accuracy_score(yt, pred); f1 = f1_score(yt, pred, average="macro")
    results.append(dict(Model=name, Family=family, Accuracy=acc, MacroF1=f1, TrainSec=secs))
    print(f"{name:15s} acc={acc:.4f}  macroF1={f1:.4f}  time={secs:.1f}s")

def chunked_forest(cls, n_total, chunk=10, **kw):
    votes = np.zeros((len(Xt), N_CLASSES), np.float32)
    for k in range(0, n_total, chunk):
        n = min(chunk, n_total - k)
        m = cls(n_estimators=n, n_jobs=-1, random_state=SEED + k, **kw).fit(Xf, yf)
        votes[:, m.classes_] += m.predict_proba(Xt) * n
        del m
    return votes.argmax(1)

# --- Naive Bayes
t = time.time(); m = GaussianNB().fit(Xf, yf); log("Naive Bayes", m.predict(Xt), time.time() - t)

# --- KNN (on the standardised raw cycle)
t = time.time(); m = KNeighborsClassifier(5, n_jobs=-1).fit(Xf[:, :T*12], yf)
log("KNN", m.predict(Xt[:, :T*12]), time.time() - t)

# --- SVM (RBF) on a PCA-reduced cycle: 420 -> 100 dims makes SVC practical
t = time.time()
pca = PCA(100, random_state=SEED).fit(Xf)
m = SVC(C=10, gamma="scale").fit(pca.transform(Xf), yf)
log("SVM (PCA100)", m.predict(pca.transform(Xt)), time.time() - t)

# --- Decision Tree
t = time.time(); m = DecisionTreeClassifier(random_state=SEED).fit(Xf, yf); log("Decision Tree", m.predict(Xt), time.time() - t)

# --- Random Forest / Extra Trees (chunked)
t = time.time(); p = chunked_forest(RandomForestClassifier, N_TREES); log("Random Forest", p, time.time() - t)
t = time.time(); p = chunked_forest(ExtraTreesClassifier,  N_TREES); log("Extra Trees",   p, time.time() - t)

In [ ]:
# --- XGBoost on the GPU (optional, slow with 390 classes)
if RUN_XGB:
    from xgboost import XGBClassifier
    t = time.time()
    m = XGBClassifier(n_estimators=150, max_depth=6, learning_rate=0.2, max_bin=64,
                      tree_method="hist", device="cuda", eval_metric="mlogloss")
    m.fit(Xf, yf)
    log("XGBoost", m.predict(Xt), time.time() - t)

## 4. Deep learning in PyTorch (GPU)
Same four architectures as your sample notebook (CNN, LSTM, CNN-LSTM, GRU) with input shape **(35, 12)** per cycle, trained
on the cycles directly, with no windowing. Speed-ups: whole dataset lives on the GPU (no DataLoader overhead), mixed precision (AMP),
batch size 256, cosine LR schedule, early stopping on the validation sessions.

Two deliberate changes from the Keras sample, because 390 classes need more capacity than 10-ish:
head width 64 → 256 (a 64-unit bottleneck is too small for 390 classes), and BatchNorm in the conv blocks. The CNN also
flattens instead of global-average-pooling: cycles are already peak-aligned, so *where* in the cycle a feature occurs is informative.

In [ ]:
import torch, torch.nn as nn
device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
torch.manual_seed(SEED); np.random.seed(SEED)
torch.backends.cudnn.benchmark = True
print("device:", device, "|", torch.cuda.get_device_name(0) if use_amp else "CPU only")

def T_(a, dt=torch.float32): return torch.as_tensor(a, dtype=dt, device=device)
Xg = T_(Xs)                        # (N, 35, 12) on GPU, ~70 MB
yg = T_(y, torch.long)
C_IN = Xs.shape[2]

In [ ]:
class CNN(nn.Module):
    def __init__(s, c_in=C_IN, n_cls=N_CLASSES, head=256, p=0.3):
        super().__init__()
        def blk(i, o, pool=True):
            L = [nn.Conv1d(i, o, 3, padding=1), nn.BatchNorm1d(o), nn.ReLU()]
            return L + ([nn.MaxPool1d(2)] if pool else [])
        s.f = nn.Sequential(*blk(c_in, 32), *blk(32, 64), *blk(64, 128, pool=False))   # 35 -> 17 -> 8
        s.h = nn.Sequential(nn.Flatten(), nn.Linear(128 * 8, head), nn.ReLU(), nn.Dropout(p), nn.Linear(head, n_cls))
    def forward(s, x): return s.h(s.f(x.transpose(1, 2)))

class RNN(nn.Module):                       # stacked LSTM or GRU
    def __init__(s, kind="LSTM", c_in=C_IN, n_cls=N_CLASSES, head=256, p=0.3):
        super().__init__()
        R = nn.LSTM if kind == "LSTM" else nn.GRU
        s.r1 = R(c_in, 64, batch_first=True); s.r2 = R(64, 128, batch_first=True)
        s.h  = nn.Sequential(nn.Linear(128, head), nn.ReLU(), nn.Dropout(p), nn.Linear(head, n_cls))
    def forward(s, x):
        x, _ = s.r1(x); x, _ = s.r2(x)
        return s.h(x[:, -1])

class CNNLSTM(nn.Module):                   # conv extracts local shape, LSTM models the sequence
    def __init__(s, c_in=C_IN, n_cls=N_CLASSES, head=256, p=0.3):
        super().__init__()
        def blk(i, o):
            return [nn.Conv1d(i, o, 3, padding=1), nn.BatchNorm1d(o), nn.ReLU(), nn.MaxPool1d(2)]
        s.f  = nn.Sequential(*blk(c_in, 32), *blk(32, 64), *blk(64, 128))               # 35 -> 17 -> 8 -> 4
        s.r1 = nn.LSTM(128, 64, batch_first=True); s.r2 = nn.LSTM(64, 128, batch_first=True)
        s.h  = nn.Sequential(nn.Dropout(p), nn.Linear(128, head), nn.ReLU(), nn.Linear(head, n_cls))
    def forward(s, x):
        x = s.f(x.transpose(1, 2)).transpose(1, 2)
        x, _ = s.r1(x); x, _ = s.r2(x)
        return s.h(x[:, -1])

BUILDERS = {
    "CNN":      lambda: CNN(),
    "LSTM":     lambda: RNN("LSTM"),
    "CNN-LSTM": lambda: CNNLSTM(),
    "GRU":      lambda: RNN("GRU"),
}

In [ ]:
@torch.no_grad()
def predict_logits(model, ids, bs=2048):
    model.eval(); out = []
    for i in range(0, len(ids), bs):
        b = T_(ids[i:i + bs], torch.long)
        with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
            out.append(model(Xg[b]).float())
    return torch.cat(out)

def accuracy_on(model, ids):
    return (predict_logits(model, ids).argmax(1) == yg[T_(ids, torch.long)]).float().mean().item()

def model_size_mb(model):
    buf = io.BytesIO(); torch.save(model.state_dict(), buf); return buf.getbuffer().nbytes / 2**20

def fit_dl(build, tr, va, epochs=80, bs=256, lr=2e-3, wd=1e-4, patience=12, verbose=True):
    model = build().to(device)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    lossf = nn.CrossEntropyLoss(label_smoothing=0.05)
    tr_t  = T_(tr, torch.long)
    hist  = {"loss": [], "train_acc": [], "val_acc": []}
    best, best_state, bad = -1, None, 0
    t0 = time.time()
    for ep in range(epochs):
        model.train()
        perm = tr_t[torch.randperm(len(tr_t), device=device)]
        tot_loss = torch.zeros((), device=device); correct = torch.zeros((), device=device)
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            if len(b) < 2: continue                      # BatchNorm needs >1 sample
            xb, yb = Xg[b], yg[b]
            with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
                out  = model(xb)
                loss = lossf(out, yb)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            tot_loss += loss.detach() * len(b); correct += (out.argmax(1) == yb).sum()   # stays on GPU, no sync
        sched.step()
        va_acc = accuracy_on(model, va)
        hist["loss"].append(tot_loss.item() / len(tr_t)); hist["train_acc"].append(correct.item() / len(tr_t)); hist["val_acc"].append(va_acc)
        if va_acc > best:
            best, bad = va_acc, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
        if verbose and (ep % 5 == 0 or ep == epochs - 1):
            print(f"  ep {ep+1:3d} loss {hist['loss'][-1]:.3f} train {hist['train_acc'][-1]:.3f} val {va_acc:.3f}")
        if bad >= patience:
            print(f"  early stop at epoch {ep+1}"); break
    model.load_state_dict(best_state)
    return model, hist, time.time() - t0

def plot_history(h, title):
    plt.figure(figsize=(9, 3.5))
    plt.plot(h["train_acc"], label="Train acc"); plt.plot(h["val_acc"], label="Val acc")
    plt.plot(h["loss"], "--", label="Train loss")
    plt.title(title); plt.xlabel("Epoch"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
dl_models, dl_hist = {}, {}
for name, build in BUILDERS.items():
    print(f"\n=== {name} ===")
    model, hist, secs = fit_dl(build, idx_train, idx_val)
    logits = predict_logits(model, idx_test)
    pred   = logits.argmax(1).cpu().numpy()
    top5   = (logits.topk(5, 1).indices.cpu().numpy() == y[idx_test][:, None]).any(1).mean()
    acc, f1 = accuracy_score(y[idx_test], pred), f1_score(y[idx_test], pred, average="macro")
    params = sum(p.numel() for p in model.parameters())
    results.append(dict(Model=name, Family="Deep", Accuracy=acc, MacroF1=f1, Top5=top5, TrainSec=secs,
                        Params=params, SizeMB=model_size_mb(model)))
    dl_models[name], dl_hist[name] = model, hist
    print(f"{name}: test acc {acc:.4f} | macro-F1 {f1:.4f} | top-5 {top5:.4f} | {params:,} params | {secs:.0f}s")
    plot_history(hist, name)

In [ ]:
# ---------- CNN-GRU: conv layers extract local shape, GRUs model the sequence ----------
class CNNGRU(nn.Module):
    def __init__(s, c_in=C_IN, n_cls=N_CLASSES, head=256, p=0.3):
        super().__init__()
        def blk(i, o):
            return [nn.Conv1d(i, o, 3, padding=1), nn.BatchNorm1d(o), nn.ReLU(), nn.MaxPool1d(2)]
        s.f  = nn.Sequential(*blk(c_in, 32), *blk(32, 64), *blk(64, 128))      # 35 -> 17 -> 8 -> 4
        s.r1 = nn.GRU(128, 64, batch_first=True)
        s.r2 = nn.GRU(64, 128, batch_first=True)
        s.h  = nn.Sequential(nn.Dropout(p), nn.Linear(128, head), nn.ReLU(), nn.Linear(head, n_cls))
    def forward(s, x):
        x = s.f(x.transpose(1, 2)).transpose(1, 2)    # (B, 4, 128)
        x, _ = s.r1(x); x, _ = s.r2(x)
        return s.h(x[:, -1])

BUILDERS["CNN-GRU"] = lambda: CNNGRU()

name = "CNN-GRU"
print(f"=== {name} ===")
model, hist, secs = fit_dl(BUILDERS[name], idx_train, idx_val)

logits = predict_logits(model, idx_test)
pred   = logits.argmax(1).cpu().numpy()
top5   = (logits.topk(5, 1).indices.cpu().numpy() == y[idx_test][:, None]).any(1).mean()
acc    = accuracy_score(y[idx_test], pred)
f1     = f1_score(y[idx_test], pred, average="macro")
params = sum(p.numel() for p in model.parameters())

results.append(dict(Model=name, Family="Deep", Accuracy=acc, MacroF1=f1, Top5=top5,
                    TrainSec=secs, Params=params, SizeMB=model_size_mb(model)))
dl_models[name], dl_hist[name] = model, hist

print(f"\n{name} -> test acc {acc:.4f} | macro-F1 {f1:.4f} | top-5 {top5:.4f} | "
      f"{params:,} params | {secs:.0f}s")
plot_history(hist, name)

## 5. Final comparison

In [ ]:
res = pd.DataFrame(results).sort_values("Accuracy", ascending=False).reset_index(drop=True)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
print(f"Split mode: {SPLIT_MODE} | test cycles: {len(idx_test)}")
display(res) if "display" in globals() else print(res.to_string())
res.to_csv(DATA_DIR / "koya_results.csv", index=False)

ax = res.set_index("Model")["Accuracy"].plot(kind="bar", figsize=(9, 4), color="#4e79a7")
ax.set_ylim(0, 1); ax.set_ylabel("Test accuracy"); ax.set_title("390-person gait identification (cycle level)")
for p in ax.patches: ax.annotate(f"{p.get_height():.3f}", (p.get_x() + p.get_width() / 2, p.get_height()), ha="center", va="bottom")
plt.xticks(rotation=30); plt.tight_layout(); plt.show()

## 6. (Optional) Session-grouped 5-fold
Each fold holds out 2 whole sessions, so test cycles never overlap with training cycles. Set `RUN_KFOLD = True` in the config cell.
Fast classical models only by default; the deep-learning loop is included but is ~5× the cost of section 4.

In [ ]:
if RUN_KFOLD:
    from sklearn.model_selection import GroupKFold
    gkf = GroupKFold(n_splits=5)
    cv = {}
    for fold, (tr, te) in enumerate(gkf.split(X_tab, y, groups=session), 1):
        for name, mk in {"KNN": lambda: KNeighborsClassifier(5, n_jobs=-1),
                         "Naive Bayes": lambda: GaussianNB()}.items():
            cols = slice(0, T * 12) if name == "KNN" else slice(None)
            m = mk().fit(X_tab[tr][:, cols], y[tr])
            cv.setdefault(name, []).append(accuracy_score(y[te], m.predict(X_tab[te][:, cols])))
        # deep models (uncomment to run):
        # for name, build in BUILDERS.items():
        #     rng = np.random.RandomState(fold); va = rng.choice(tr, len(tr)//10, replace=False)
        #     mdl, _, _ = fit_dl(build, np.setdiff1d(tr, va), va, verbose=False)
        #     cv.setdefault(name, []).append(accuracy_on(mdl, te))
    print(pd.DataFrame({k: [np.mean(v), np.std(v)] for k, v in cv.items()}, index=["mean", "std"]).T)